In [1]:
from __future__ import division, print_function
import sys, os, glob, time, warnings, gc
import numpy as np
# import matplotlib
# matplotlib.use("Agg")
import matplotlib.pyplot as plt
from astropy.table import Table, vstack, hstack, join
import fitsio
# from astropy.io import fits

from scipy.optimize import curve_fit

In [2]:
params = {'legend.fontsize': 'large',
         'axes.labelsize': 'large',
         'axes.titlesize':'large',
         'xtick.labelsize':'large',
         'ytick.labelsize':'large',
         'figure.facecolor':'w'} 
plt.rcParams.update(params)

In [3]:
# main = Table(fitsio.read('/global/cfs/cdirs/desi/users/rongpu/spectro/fugu/main_cumulative_lrg.fits'))
main = Table(fitsio.read('/Users/rongpu/Documents/Data/desi_data/fugu/main_cumulative_lrg.fits'))
main['EFFTIME_ELG'] = 8.60 * main['TSNR2_ELG']
main['EFFTIME_LRG'] = 12.15 * main['TSNR2_LRG']

# Remove FIBERSTATUS!=0 fibers
mask = main['COADD_FIBERSTATUS']==0
print('FIBERSTATUS',np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
main = main[mask]

# Remove "no data" fibers
mask = main['ZWARN'] & 2**9==0
print('No data', np.sum(mask), np.sum(~mask), np.sum(~mask)/len(mask))
main = main[mask]

FIBERSTATUS 338266 7165 0.020742203218587794
No data 338265 1 2.9562533627382e-06


In [19]:
ls -lh /Users/rongpu/Documents/Data/desi_data/fugu

total 691016
-rw-rw----  1 rongpu  staff    98M Mar 10 08:20 main_cumulative_lrg.fits
drwxr-xr-x  4 rongpu  staff   128B May 10 17:37 misc/
drwxrws---  9 rongpu  staff   288B Mar  8 17:13 old/
-rw-rw----  1 rongpu  staff   8.3M Mar 10 08:18 sv1_1x_depth_lrg.fits
-rw-rw----  1 rongpu  staff   2.6M Mar 10 08:20 sv1_4x_depth_lrg.fits
-rw-rw----  1 rongpu  staff    14M Mar 10 08:18 sv1_cumulative_lrg.fits
-rw-rw----  1 rongpu  staff   1.6M Mar 10 08:20 sv1_lowspeed_lrg.fits
-rw-rw----  1 rongpu  staff   163M Mar 10 08:22 sv1_perexp_lrg.fits
-rw-rw----  1 rongpu  staff    48M Mar 10 08:19 sv3_cumulative_lrg.fits
-rw-r-----  1 rongpu  staff   1.5M Feb 16 20:28 tiles-main-20220216.ecsv


In [18]:
main = Table(fitsio.read('/Users/rongpu/Documents/Data/desi_data/fugu/main_cumulative_bgs.fits'))



OSError: FITSIO status = 104: could not open the named file
failed to find or open the following file: (ffopen)
/Users/rongpu/Documents/Data/desi_data/fugu/main_cumulative_bgs.fits


In [7]:
ls -lh /Users/rongpu/Documents/Data/desi_data/fugu

total 691016
-rw-rw----  1 rongpu  staff    98M Mar 10 08:20 main_cumulative_lrg.fits
drwxr-xr-x  4 rongpu  staff   128B May 10 17:37 misc/
drwxrws---  9 rongpu  staff   288B Mar  8 17:13 old/
-rw-rw----  1 rongpu  staff   8.3M Mar 10 08:18 sv1_1x_depth_lrg.fits
-rw-rw----  1 rongpu  staff   2.6M Mar 10 08:20 sv1_4x_depth_lrg.fits
-rw-rw----  1 rongpu  staff    14M Mar 10 08:18 sv1_cumulative_lrg.fits
-rw-rw----  1 rongpu  staff   1.6M Mar 10 08:20 sv1_lowspeed_lrg.fits
-rw-rw----  1 rongpu  staff   163M Mar 10 08:22 sv1_perexp_lrg.fits
-rw-rw----  1 rongpu  staff    48M Mar 10 08:19 sv3_cumulative_lrg.fits
-rw-r-----  1 rongpu  staff   1.5M Feb 16 20:28 tiles-main-20220216.ecsv


In [8]:
t1 = Table(fitsio.read('/Users/rongpu/Documents/Data/desi_data/fugu/sv1_1x_depth_lrg.fits'))
t2 = Table(fitsio.read('/Users/rongpu/Documents/Data/desi_data/fugu/sv1_4x_depth_lrg.fits'))

In [17]:
len(np.unique(main['TILEID']))

305

In [20]:
len(np.unique(t1['TILEID']))

3

In [21]:
len(np.unique(t2['TILEID']))

3

In [11]:
t1

TARGETID,Z,ZERR,ZWARN,CHI2,SPECTYPE,DELTACHI2,PETAL_LOC,DEVICE_LOC,LOCATION,FIBER,COADD_FIBERSTATUS,TARGET_RA,TARGET_DEC,MORPHTYPE,EBV,FLUX_G,FLUX_R,FLUX_Z,FLUX_W1,FLUX_W2,FIBERFLUX_G,FIBERFLUX_R,FIBERFLUX_Z,MASKBITS,PARALLAX,PHOTSYS,SV1_DESI_TARGET,SV1_BGS_TARGET,DESI_TARGET,BGS_TARGET,TILEID,COADD_NUMEXP,COADD_EXPTIME,COADD_NUMNIGHT,COADD_NUMTILE,TSNR2_ELG,TSNR2_BGS,TSNR2_QSO,TSNR2_LRG,fn,subset,lrg_mask,main_lrg
int64,float64,float64,int64,float64,str6,float64,int16,int32,int64,int32,int32,float64,float64,str4,float32,float32,float32,float32,float32,float32,float32,float32,float32,int16,float32,str1,int64,int64,int64,int64,int32,int16,float32,int16,int16,float32,float32,float32,float32,str58,int64,int16,bool
39627634551823006,-0.0019957014509158224,4.131159871613317e-48,1570,8.999999999999996e+99,STAR,1.942668892225729e+84,0,526,526,365,512,36.13504280127578,-6.142154477217699,REX,0.03161644,1.7430092,5.966546,15.021786,41.361485,16.829456,0.5679733,1.9442462,4.894967,0,0.0,S,1152921504673955905,262148,0,0,80605,0,0.0,0,0,192.3893,55356.32,24.359364,393.9989,fuji/tiles/1x_depth/80605/1/redrock-0-80605-1xsubset1.fits,1,0,False
39627634551823006,-0.0019957014509158224,4.131159871613317e-48,1570,8.999999999999996e+99,STAR,1.942668892225729e+84,0,526,526,365,512,36.13504280127578,-6.142154477217699,REX,0.03161644,1.7430092,5.966546,15.021786,41.361485,16.829456,0.5679733,1.9442462,4.894967,0,0.0,S,1152921504673955905,262148,0,0,80605,0,0.0,0,0,166.88437,37681.117,25.826433,282.87552,fuji/tiles/1x_depth/80605/2/redrock-0-80605-1xsubset2.fits,2,0,False
39627634551823006,-0.0019957014509158224,4.131159871613317e-48,1570,8.999999999999996e+99,STAR,1.942668892225729e+84,0,526,526,365,512,36.13504280127578,-6.142154477217699,REX,0.03161644,1.7430092,5.966546,15.021786,41.361485,16.829456,0.5679733,1.9442462,4.894967,0,0.0,S,1152921504673955905,262148,0,0,80605,0,0.0,0,0,108.14054,6620.658,28.492031,75.175064,fuji/tiles/1x_depth/80605/4/redrock-0-80605-1xsubset4.fits,4,0,False
39627634551823006,-0.0019957014509158224,4.131159871613317e-48,1570,8.999999999999996e+99,STAR,1.942668892225729e+84,0,526,526,365,512,36.13504280127578,-6.142154477217699,REX,0.03161644,1.7430092,5.966546,15.021786,41.361485,16.829456,0.5679733,1.9442462,4.894967,0,0.0,S,1152921504673955905,262148,0,0,80605,0,0.0,0,0,197.63779,56338.535,25.101334,404.61053,fuji/tiles/1x_depth/80605/5/redrock-0-80605-1xsubset5.fits,5,0,False
39627634551823006,-0.0019957014509158224,4.131159871613317e-48,1570,8.999999999999996e+99,STAR,1.942668892225729e+84,0,526,526,365,512,36.13504280127578,-6.142154477217699,REX,0.03161644,1.7430092,5.966546,15.021786,41.361485,16.829456,0.5679733,1.9442462,4.894967,0,0.0,S,1152921504673955905,262148,0,0,80605,0,0.0,0,0,154.20793,30513.227,27.764103,229.45398,fuji/tiles/1x_depth/80605/3/redrock-0-80605-1xsubset3.fits,3,0,False
39627634551824898,0.354928167900012,5.573504505970296e-05,0,8091.289611555636,GALAXY,896.7260583415627,0,525,525,398,0,36.18946453136912,-6.149115843122294,DEV,0.03287473,4.346976,21.593157,48.598835,74.80693,51.556156,1.1649392,5.7867155,13.023925,0,0.0,S,1152921504715898985,131074,0,0,80605,2,1800.0598,2,1,121.45346,7380.7705,30.528109,84.13693,fuji/tiles/1x_depth/80605/3/redrock-0-80605-1xsubset3.fits,3,0,False
39627634551824898,0.35502735211788267,5.7826563665040775e-05,0,8179.911092482507,GALAXY,895.3923364318907,0,525,525,398,0,36.18946453136912,-6.149115843122294,DEV,0.03287473,4.346976,21.593157,48.598835,74.80693,51.556156,1.1649392,5.7867155,13.023925,0,0.0,S,1152921504715898985,131074,0,0,80605,2,1357.9677,1,1,107.37782,6307.8164,26.632772,72.505226,fuji/tiles/1x_depth/80605/5/redrock-0-80605-1xsubset5.fits,5,0,False
39627634551824898,0.35486475866865,5.471015349640114e-05,0,8289.768116548657,GALAXY,782.5249964594841,0,525,525,398,0,36.18946453136912,-6.149115843122294,DEV,0.03287473,4.346976,21.593157,48.598835,74.80693,51.556156,1.1649392,5.7867155,13.023925,0,0.0,S,1152921504715898985,131074,0,0,80605,2,180

In [12]:
subsamp_dict = {'80605': [[74781, 73702],
  [67975, 74782],
  [68292, 74779],
  [68291, 68290],
  [74783, 74780]],
 '80606': [[68630, 67968], [67971, 68812], [67969, 68813, 67970]],
 '80607': [[67768, 68844],
  [68027, 68845],
  [67767, 68028, 67766],
  [68847, 67744],
  [67765, 68846]],
 '80608': [[68025, 68328],
  [67770, 68026],
  [67771, 68327],
  [67769, 68024],
  [68842, 68023],
  [68491, 68841]],
 '80609': [[68489, 67781],
  [67784, 68338],
  [68336, 68490],
  [68064, 68334],
  [68340, 67783],
  [68063, 68065]],
 '80610': [[68333, 68477],
  [68332, 68331],
  [68330, 68042],
  [68488, 68041],
  [68040, 75116]],

 '80620': [[68677, 68851], [68676, 68850]],
 '80678': [[71878, 72681],
  [74946, 74943],
  [72505, 71876],
  [72504, 71877],
  [74944, 74945]],
 '80690': [[71892, 76294],
  [75840, 74802, 75841],
  [75838, 72515, 72692],
  [75842, 75839, 76293],
  [74801, 72514],
  [74803, 76295]],
 '80695': [[80944, 79553], [79552, 80943]],
 '80699': [[72519, 71481], [72518, 71618], [79447, 71619]],
 '80711': [[75862, 75860, 74462, 75861],
  [75859, 72714, 86518],
  [74831, 74832],
  [72715, 74829, 74830]]}

exp_list = sorted([i for k in subsamp_dict.values() for j in k for i in j])
print(len(exp_list), len(np.unique(exp_list)))

113 113


In [13]:
exp_list

[67744,
 67765,
 67766,
 67767,
 67768,
 67769,
 67770,
 67771,
 67781,
 67783,
 67784,
 67968,
 67969,
 67970,
 67971,
 67975,
 68023,
 68024,
 68025,
 68026,
 68027,
 68028,
 68040,
 68041,
 68042,
 68063,
 68064,
 68065,
 68290,
 68291,
 68292,
 68327,
 68328,
 68330,
 68331,
 68332,
 68333,
 68334,
 68336,
 68338,
 68340,
 68477,
 68488,
 68489,
 68490,
 68491,
 68630,
 68676,
 68677,
 68812,
 68813,
 68841,
 68842,
 68844,
 68845,
 68846,
 68847,
 68850,
 68851,
 71481,
 71618,
 71619,
 71876,
 71877,
 71878,
 71892,
 72504,
 72505,
 72514,
 72515,
 72518,
 72519,
 72681,
 72692,
 72714,
 72715,
 73702,
 74462,
 74779,
 74780,
 74781,
 74782,
 74783,
 74801,
 74802,
 74803,
 74829,
 74830,
 74831,
 74832,
 74943,
 74944,
 74945,
 74946,
 75116,
 75838,
 75839,
 75840,
 75841,
 75842,
 75859,
 75860,
 75861,
 75862,
 76293,
 76294,
 76295,
 79447,
 79552,
 79553,
 80943,
 80944,
 86518]

In [14]:
subsamp_dict = {'80605': [[73702, 74779, 68291, 74780, 68292, 74782, 67972]],
 '80606': [[67970, 68813, 68285, 68289, 68630, 67968, 67971]],
 '80607': [[67767, 68028, 68847, 68664, 68845, 68846],
  [67765, 68662, 68663, 68027, 67766, 68666]],
 '80608': [[68842, 69252, 68023, 68024, 68661],
  [69253, 68491, 67769, 69249, 67770, 68328, 67771],
  [68660, 68327, 68317, 68025, 68841]],
 '80609': [[68490, 68339, 68337, 68334, 68065, 68336],
  [67781, 68063, 68064, 67783, 68340, 68489, 68338]],
 '80610': [[68332, 68478, 68041, 68330, 68488, 68683, 75114],
  [75113, 68477, 68040, 68333, 68331, 68042, 75116]],
 '80613': [[69226, 68658, 69230, 69228],
  [81833, 81834],
  [81835, 69227, 68657],
  [69229, 81832, 69225, 68659, 81831]]}

In [15]:
exp_list = sorted([i for k in subsamp_dict.values() for j in k for i in j])
print(len(exp_list), len(np.unique(exp_list)))

84 84
